# F00 ASSETFORGE — Phase 2 : Image Generation

**Modèle :** FLUX.1-schnell (Apache 2.0, gratuit, HuggingFace)

**Runtime :** GPU T4 (Kaggle gratuit)

**Process :**
1. Télécharger `prompts_manifest.json` depuis l'URL passée en variable Kaggle
2. Pour chaque image : construire le prompt → générer avec FLUX.1-schnell → sauvegarder
3. Zipper le dossier output

**Variables d'environnement Kaggle :**
- `MANIFEST_URL` : URL de téléchargement du prompts_manifest.json
- `KAGGLE_DATASET` : (optionnel) dataset de sortie personnalisé


In [ ]:
# === INSTALL ===
!pip install -q diffusers transformers accelerate torch
print('Dependencies installed')

In [ ]:
# === LOAD MODEL ===
# FLUX.1-schnell : le plus rapide, ~2-4s/image sur T4
# Licence Apache 2.0 — usage commercial OK

if not torch.cuda.is_available():
    print('ERROR: GPU NOT AVAILABLE!')
    print(f'PyTorch: {torch.__version__}')
    print('This notebook REQUIRES GPU. Enable it in Kaggle:')
    print('Settings > Accelerator > GPU T4 x2')
    raise RuntimeError('GPU required but not available. Check Kaggle accelerator settings.')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.float16
)
pipe = pipe.to('cuda')

# Optimisations pour T4 (16GB VRAM)
pipe.enable_attention_slicing()
try:
    pipe.enable_model_cpu_offload()
except Exception:
    pass

print('FLUX.1-schnell loaded on GPU')

In [ ]:
# === LOAD MANIFEST ===
# The manifest is embedded directly by the GitHub Actions workflow
# No dataset dependency — the workflow injects the JSON before pushing
import json, os, glob

manifest = None

# Method 1: Embedded manifest (injected by workflow)
try:
    manifest = EMBEDDED_MANIFEST  # type: ignore
    print('Using embedded manifest (injected by workflow)')
except NameError:
    pass

# Method 2: Search in Kaggle input datasets
if manifest is None:
    candidates = glob.glob('/kaggle/input/*/prompts_manifest.json')
    if candidates:
        with open(candidates[0]) as f:
            manifest = json.load(f)
        print(f'Found manifest in dataset: {candidates[0]}')

# Method 3: Search for any JSON with meta+images keys
if manifest is None:
    candidates = glob.glob('/kaggle/input/**/*.json', recursive=True)
    for c in candidates:
        try:
            with open(c) as f:
                d = json.load(f)
            if 'images' in d and 'meta' in d:
                manifest = d
                print(f'Found manifest-like JSON: {c}')
                break
        except:
            pass

if manifest is None:
    raise ValueError('prompts_manifest.json not found. '
                     'Expected embedded EMBEDDED_MANIFEST, /kaggle/input/, or any input JSON.')

meta = manifest['meta']
images = manifest['images']
print(f'Mode: {meta["mode"]}')
print(f'Format: {meta["format"]}')
print(f'Total images: {meta["total_images"]}')
print(f'Images to generate: {len(images)}')

In [ ]:
# === LOAD MODEL ===
# FLUX.1-schnell : le plus rapide, ~2-4s/image sur T4
# Licence Apache 2.0 — usage commercial OK
pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell',
    torch_dtype=torch.float16
)
pipe = pipe.to('cuda')

# Optimisations pour T4 (16GB VRAM)
pipe.enable_attention_slicing()
try:
    pipe.enable_model_cpu_offload()
except Exception:
    pass  # Pas toujours nécessaire sur T4

print('FLUX.1-schnell loaded on GPU')

In [ ]:
# === GENERATE IMAGES ===
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dimensions selon le format
if meta['format'] == 'VERTICAL':
    WIDTH, HEIGHT = 1080, 1920
else:
    WIDTH, HEIGHT = 1920, 1080

# FLUX.1-schnell fonctionne en 4 steps (schnell = fast)
NUM_STEPS = 4
GUIDANCE = 0.0  # schnell n'utilise pas de guidance scale

generated = []
failed = []

for i, img_spec in enumerate(images):
    filename = img_spec['filename']
    prompt = img_spec['prompt']
    overlay = img_spec.get('overlay', 'defaut')
    intensite = img_spec.get('intensite', 2)
    
    # Construire le prompt final avec contraintes de style
    style_constraints = (
        f'No text, no watermark, no logo in the image. '
        f'High contrast, clear composition. '
        f'Format: {meta["format"].lower()} {WIDTH}x{HEIGHT}. '
        f'Consistent visual style across all images.'
    )
    full_prompt = f'{prompt}. {style_constraints}'
    
    print(f'[{i+1}/{len(images)}] Generating {filename}...')
    t0 = time.time()
    
    try:
        result = pipe(
            prompt=full_prompt,
            width=WIDTH,
            height=HEIGHT,
            num_inference_steps=NUM_STEPS,
            guidance_scale=GUIDANCE,
        )
        image = result.images[0]
        image.save(OUTPUT_DIR / filename)
        elapsed = time.time() - t0
        print(f'  ✓ {filename} ({elapsed:.1f}s)')
        generated.append(filename)
    except Exception as e:
        print(f'  ✗ FAILED: {e}')
        failed.append({'filename': filename, 'error': str(e)})
    
    # Libérer la mémoire entre les générations
    torch.cuda.empty_cache()

print(f'\n=== RÉSUMÉ ===')
print(f'Générées: {len(generated)}/{len(images)}')
if failed:
    print(f'Échouées: {len(failed)}')
    for f in failed:
        print(f'  - {f["filename"]}: {f["error"][:80]}')

In [ ]:
# === ZIP OUTPUT ===
import zipfile

ZIP_PATH = '/kaggle/working/f00_images.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for img_file in sorted(OUTPUT_DIR.iterdir()):
        if img_file.is_file():
            zf.write(img_file, img_file.name)

print(f'Output zipped: {ZIP_PATH}')
print(f'Files in zip: {len(generated)}')
print(f'Total size: {os.path.getsize(ZIP_PATH) / 1024 / 1024:.1f} MB')

In [ ]:
# === SAVE METADATA ===
metadata = {
    'model': 'FLUX.1-schnell',
    'format': meta['format'],
    'mode': meta['mode'],
    'total_requested': len(images),
    'total_generated': len(generated),
    'failed': failed,
    'generation_params': {
        'steps': NUM_STEPS,
        'guidance': GUIDANCE,
        'width': WIDTH,
        'height': HEIGHT,
    },
}

with open('/kaggle/working/generation_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Metadata saved.')
print(f'\nF00 ASSETFORGE Phase 2 — COMPLETE')
print(f'Images: {len(generated)}/{len(images)}')